# Spectra of J_par and J_perp across the beta scan

One notebook for what used to be three (`JparSpectrum`, `JperpSpectrum`, `JperpSpectrum40`, which differed only in the
component and the snapshot; they are in `archive/jpar_jperp_spectra/`). Choose `COMPONENT` and `TIME` in the settings
cell. The figure is saved as `J<component>_Spectrum_beta_scan_t<TIME>.png` in `results/jpar_jperp/JparJperpSpectrum/`,
the same names the three notebooks used.

* J_par = J . B / |B|; its spectrum is computed directly.
* J_perp: spectrum of |J|^2 (sum of the Jx, Jy, Jz spectra) minus the J_par spectrum, as in the old notebooks.

In [ ]:
import matplotlib.pyplot as plt

from pta import config
from pta.fields import current_density
from pta.plotting import save_figure, use_style
from pta.spectra import parallel_perpendicular_spectra

use_style()
RESULTS = config.results_dir("jpar_jperp", "JparJperpSpectrum")

In [ ]:
COMPONENT = "par"                     # "par" for J_par, "perp" for J_perp
TIME = 200                            # snapshot (the old notebooks: J_par at t = 200, J_perp at t = 200 and t = 40)
BETAS_P = [0.0625, 1, 16]             # proton beta of the runs (the beta_scan layout in pta.toml)
K_RANGE_SHOWN = (5e-2, 30)
POWER_RANGE_SHOWN = (1e-6, 1)

In [ ]:
spectra = {}
for beta_p in BETAS_P:
    run = config.run("beta_scan", beta_p=beta_p)
    B = run.load_many(["Bx", "By", "Bz"], TIME)
    J = current_density(*B, run.dx, run.dy)
    k, total, parallel, perpendicular = parallel_perpendicular_spectra(J, B, run.dx, run.dy)
    spectra[beta_p] = (k, parallel if COMPONENT == "par" else perpendicular)

In [ ]:
fig, ax = plt.subplots()
for beta_p, (k, spectrum) in spectra.items():
    ax.loglog(k, spectrum, label=rf"$\beta_p = {beta_p}$")
ax.set_xlabel(r"$k$")
ax.set_ylabel(r"$P(k)$")
ax.set_xlim(*K_RANGE_SHOWN)
ax.set_ylim(*POWER_RANGE_SHOWN)
ax.legend()
save_figure(fig, RESULTS, f"J{COMPONENT}_Spectrum_beta_scan_t{TIME}")
plt.show()